# 🤖 Sentence Embeddings: From Theory to Practical Applications

## 📚 Overview
Sentence embeddings transform textual information into **numerical vectors that capture semantic meaning**. 

This notebook demonstrates:

- Basic embedding generation using state-of-the-art models

- Visualization techniques for understanding semantic relationships

- Retrieval-Augmented Generation (RAG) with multi-stage retrieval

- Real-world applications in search, clustering, and recommendation systems

## 🎯 Key Concepts Explained
- **Embeddings**: Dense vector representations of text (typically 384-1024 dimensions)

- **Semantic Similarity**: How similar two pieces of text are in meaning, not just keywords

- **Cosine Similarity**: Measure of similarity between two non-zero vectors

- **RAG Pipeline**: Retrieve relevant information → Generate informed responses


## 🛠️ Installation
We'll install essential libraries for embeddings, visualization, and machine learning.

In [ ]:
!pip install -qU sentence-transformers numpy matplotlib scikit-learn FlagEmbedding

## 🔤 Generating Sentence Embeddings

### Why Embeddings Matter

Embeddings convert text into vectors that:

- Capture semantic meaning (similar words/phrases have similar vectors)

- Enable mathematical operations (add, subtract, compare)

- Power modern AI applications (search, recommendations, chatbots)

### Model Selection Guide

- **all-MiniLM-L6-v2**: Fast, 384 dimensions, good balance of speed/accuracy

- **all-mpnet-base-v2**: Slower but more accurate, 768 dimensions

- **paraphrase-MiniLM-L6-v2**: Optimized for paraphrase detection

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Initialize the model (using a popular pre-trained model)
model = SentenceTransformer('all-MiniLM-L6-v2')  # You can use other models like 'paraphrase-MiniLM-L6-v2'

# Your source texts
source_texts = [
    "The capital of France is Paris",
    "PyTorch is a machine learning framework based on the Torch library.",
    "When you feel very happy, it often reflects a sense of joy and satisfaction in your life.",
    "You're a wizard, Harry.",
    "The Eiffel Tower is in Paris, France.",
    "Paris is the capital city of France.",
    "France is known for its wine and cuisine.",
    "The Louvre Museum houses the Mona Lisa."
    "Space, the final frontier.",
    "Mr. Dursley always sat with his back to the window in his office on the ninth floor.",
    "Joining Slack Channels: You will receive an invite via email. Be sure to join relevant channels to stay informed and engaged.",
    "Finding Coffee Spots: For your caffeine fix, head to the break room's coffee machine or cross the street to the café for artisan coffee.",
    "Team-Building Activities: We foster team spirit with monthly outings and weekly game nights. Feel free to suggest new activity ideas anytime!",
    "Working Hours Flexibility: We prioritize work-life balance. While our core hours are 9 AM to 5 PM, we offer flexibility to adjust as needed.",
]

# Method 1: Embed all sentences at once (most efficient)
print("Method 1: Embedding all sentences at once")
embeddings = model.encode(source_texts)
print(f"Shape of embeddings: {embeddings.shape}")
print(f"Number of sentences: {len(source_texts)}")
print(f"Embedding dimension: {embeddings.shape[1]}")
print()

# Method 2: Embed one by one (as requested)
print("Method 2: Embedding sentences one by one")
individual_embeddings = []

for i, text in enumerate(source_texts):
    # Encode the single sentence
    embedding = model.encode(text)
    individual_embeddings.append(embedding)

    print(f"Sentence {i+1}:")
    print(f"  Text: {text[:50]}..." if len(text) > 50 else f"  Text: {text}")
    print(f"  Embedding shape: {embedding.shape}")
    print(f"  Embedding (first 5 values): {embedding[:5]}")
    print()

# Convert to numpy array for easier manipulation
individual_embeddings = np.array(individual_embeddings)
print(f"Final shape of individual embeddings array: {individual_embeddings.shape}")

# Verify both methods produce the same results
print("\nVerification - Checking if both methods produce same results:")
print(f"Are the embeddings from both methods equal? {np.allclose(embeddings, individual_embeddings)}")

# Save embeddings if needed
print("\nSaving embeddings for later use...")
np.save('sentence_embeddings.npy', individual_embeddings)

# To calculate similarity between sentences
print("\nCalculating similarity between first two sentences:")
from sklearn.metrics.pairwise import cosine_similarity

# Calculate cosine similarity between first two sentences
sim_matrix = cosine_similarity([individual_embeddings[0]], [individual_embeddings[1]])
print(f"Cosine similarity between sentence 1 and 2: {sim_matrix[0][0]:.4f}")

## 📊 Visualizing Embeddings

### Why Visualization Matters

1- Understand semantic relationships between texts

2- Identify clusters of similar content

3- Debug embedding quality

4- Communicate results to stakeholders

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import plotly.graph_objects as go
import plotly.express as px
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Initialize the model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Your source texts
source_texts = [
    "The capital of France is Paris",
    "PyTorch is a machine learning framework based on the Torch library.",
    "The average cat lifespan is between 13-17 years",
    "You're a wizard, Harry.",
    "The Eiffel Tower is in Paris, France.",
    "Paris is the capital city of France.",
    "France is known for its wine and cuisine.",
    "The Louvre Museum houses the Mona Lisa."
    "Space, the final frontier.",
    "I'm going to make him an offer he can't refuse.",
    "Joining Slack Channels: You will receive an invite via email. Be sure to join relevant channels to stay informed and engaged.",
    "Finding Coffee Spots: For your caffeine fix, head to the break room's coffee machine or cross the street to the café for artisan coffee.",
    "Team-Building Activities: We foster team spirit with monthly outings and weekly game nights. Feel free to suggest new activity ideas anytime!",
    "Working Hours Flexibility: We prioritize work-life balance. While our core hours are 9 AM to 5 PM, we offer flexibility to adjust as needed.",
]

# Create categories for coloring (based on content)
categories = [
    "Geography",           # 0: France capital
    "Technology",          # 1: PyTorch
    "Animals",             # 2: Cat lifespan
    "Movie Quote",         # 3: Harry Potter
    "Geography",           # 4: Eiffel Tower (ajouté)
    "Geography",           # 5: Paris is capital (ajouté)
    "Geography",           # 6: France wine (ajouté)
    "Art",                 # 7: Louvre Museum (ajouté)
    "Movie Quote",         # 8: Star Trek
    "Movie Quote",         # 9: Godfather
    "Workplace",           # 10: Slack channels
    "Workplace",           # 11: Coffee spots
    "Workplace",           # 12: Team activities
    "Workplace",           # 13: Working hours
]

# Get embeddings
embeddings = model.encode(source_texts)
print(f"Original embedding shape: {embeddings.shape}")
print(f"Reducing from {embeddings.shape[1]} dimensions to 3D...")

# Method 1: PCA for dimensionality reduction
pca = PCA(n_components=3)
embeddings_3d_pca = pca.fit_transform(embeddings)
print(f"\nPCA explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained by 3 components: {sum(pca.explained_variance_ratio_):.2%}")

# Method 2: t-SNE for dimensionality reduction (better for visualization)
tsne = TSNE(n_components=3, random_state=42, perplexity=min(5, len(embeddings)-1))
embeddings_3d_tsne = tsne.fit_transform(embeddings)
print("\nt-SNE transformation complete.")

# Function to create matplotlib 3D plot
def plot_matplotlib_3d(embeddings_3d, title, method="PCA"):
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')

    # Color mapping for categories
    unique_categories = list(set(categories))
    colors = plt.cm.Set2(np.linspace(0, 1, len(unique_categories)))
    color_map = {cat: colors[i] for i, cat in enumerate(unique_categories)}

    # Plot each point
    for i, (x, y, z) in enumerate(embeddings_3d):
        category = categories[i]
        ax.scatter(x, y, z,
                   color=color_map[category],
                   s=200,
                   alpha=0.8,
                   edgecolors='black',
                   linewidth=1.5)

        # Add text label (abbreviated for clarity)
        label = f"{i}"
        ax.text(x, y, z, label, fontsize=12, fontweight='bold')

    ax.set_xlabel('Dimension 1')
    ax.set_ylabel('Dimension 2')
    ax.set_zlabel('Dimension 3')
    ax.set_title(f'3D Visualization of Sentence Embeddings ({method})', fontsize=16, pad=20)

    # Add legend
    legend_elements = [plt.Line2D([0], [0], marker='o', color='w',
                                  markerfacecolor=color_map[cat],
                                  markersize=10,
                                  label=cat,
                                  markeredgecolor='black')
                      for cat in unique_categories]
    ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.05, 1))

    # Add grid
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Print sentence mapping
    print(f"\n{title} - Sentence Mapping:")
    for i, (text, cat) in enumerate(zip(source_texts, categories)):
        print(f"{i}: [{cat}] {text[:50]}...")
    print()

# Function to create interactive Plotly 3D plot
def plot_interactive_3d(embeddings_3d, title, method="PCA"):
    # Create hover text
    hover_texts = []
    for i, (text, cat) in enumerate(zip(source_texts, categories)):
        hover_texts.append(f"<b>Sentence {i}</b><br>Category: {cat}<br>Text: {text[:100]}...")

    # Create the figure
    fig = go.Figure(data=[go.Scatter3d(
        x=embeddings_3d[:, 0],
        y=embeddings_3d[:, 1],
        z=embeddings_3d[:, 2],
        mode='markers+text',
        marker=dict(
            size=10,
            color=[categories.index(cat) for cat in categories],  # Color by category index
            colorscale='Viridis',
            opacity=0.8,
            line=dict(width=2, color='DarkSlateGrey')
        ),
        text=[str(i) for i in range(len(source_texts))],
        textposition="top center",
        hovertext=hover_texts,
        hoverinfo='text'
    )])

    # Update layout
    fig.update_layout(
        title=dict(
            text=f'Interactive 3D Visualization of Sentence Embeddings ({method})',
            font=dict(size=20)
        ),
        scene=dict(
            xaxis_title='Dimension 1',
            yaxis_title='Dimension 2',
            zaxis_title='Dimension 3'
        ),
        width=1000,
        height=800,
        showlegend=False
    )

    # Add annotations for each point
    for i, (x, y, z) in enumerate(embeddings_3d):
        fig.add_annotation(
            x=x, y=y, z=z,
            text=str(i),
            showarrow=False,
            font=dict(size=12, color="black"),
            yshift=10
        )

    fig.show()

    # Create a separate 2D scatter matrix for better cluster analysis
    fig_2d = px.scatter_matrix(
        x=embeddings_3d[:, 0],
        y=embeddings_3d[:, 1],
        color=categories,
        title=f'2D Scatter Matrix ({method})',
        labels={'color': 'Category'},
        hover_data={'index': list(range(len(source_texts)))}
    )
    fig_2d.update_traces(diagonal_visible=False)
    fig_2d.show()

# Function to calculate and display cluster distances
def analyze_clusters(embeddings_3d):
    print("\n" + "="*60)
    print("CLUSTER ANALYSIS")
    print("="*60)

    from scipy.spatial.distance import cdist

    # Group by category
    category_points = {}
    for i, cat in enumerate(categories):
        if cat not in category_points:
            category_points[cat] = []
        category_points[cat].append(embeddings_3d[i])

    print("\nIntra-cluster distances (within same category):")
    for cat, points in category_points.items():
        if len(points) > 1:
            points_array = np.array(points)
            distances = cdist(points_array, points_array)
            avg_distance = np.mean(distances[np.triu_indices(len(points), 1)])
            print(f"  {cat}: {len(points)} points, avg distance = {avg_distance:.4f}")
        else:
            print(f"  {cat}: {len(points)} point (no intra-cluster distance)")

    print("\nInter-cluster distances (between categories):")
    categories_list = list(category_points.keys())
    for i in range(len(categories_list)):
        for j in range(i+1, len(categories_list)):
            cat1, cat2 = categories_list[i], categories_list[j]
            points1 = np.array(category_points[cat1])
            points2 = np.array(category_points[cat2])

            distances = cdist(points1, points2)
            avg_distance = np.mean(distances)
            print(f"  {cat1} ↔ {cat2}: avg distance = {avg_distance:.4f}")

# Plot PCA results
print("\n" + "="*60)
print("VISUALIZING WITH PCA")
print("="*60)
plot_matplotlib_3d(embeddings_3d_pca, "PCA Visualization", method="PCA")
analyze_clusters(embeddings_3d_pca)

# Plot t-SNE results
print("\n" + "="*60)
print("VISUALIZING WITH t-SNE")
print("="*60)
plot_matplotlib_3d(embeddings_3d_tsne, "t-SNE Visualization", method="t-SNE")
analyze_clusters(embeddings_3d_tsne)

# Create interactive plots (uncomment to use)
print("\n" + "="*60)
print("INTERACTIVE VISUALIZATIONS")
print("="*60)
print("Creating interactive plots...")

# For interactive plots (Plotly), uncomment these lines:
# plot_interactive_3d(embeddings_3d_pca, "PCA", method="PCA")
# plot_interactive_3d(embeddings_3d_tsne, "t-SNE", method="t-SNE")

# Alternative: Simple 2D visualization for comparison
print("\n" + "="*60)
print("2D VISUALIZATION FOR COMPARISON")
print("="*60)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# PCA 2D
pca_2d = PCA(n_components=2)
embeddings_2d_pca = pca_2d.fit_transform(embeddings)

unique_categories = list(set(categories))
colors = plt.cm.Set2(np.linspace(0, 1, len(unique_categories)))
color_map = {cat: colors[i] for i, cat in enumerate(unique_categories)}

ax1 = axes[0]
for i, (x, y) in enumerate(embeddings_2d_pca):
    category = categories[i]
    ax1.scatter(x, y, color=color_map[category], s=150, alpha=0.8, edgecolors='black')
    ax1.text(x, y, str(i), fontsize=11, fontweight='bold', ha='center', va='center')

ax1.set_xlabel('PC1')
ax1.set_ylabel('PC2')
ax1.set_title('2D PCA Visualization', fontsize=14)
ax1.grid(True, alpha=0.3)

# t-SNE 2D
tsne_2d = TSNE(n_components=2, random_state=42, perplexity=min(5, len(embeddings)-1))
embeddings_2d_tsne = tsne_2d.fit_transform(embeddings)

ax2 = axes[1]
for i, (x, y) in enumerate(embeddings_2d_tsne):
    category = categories[i]
    ax2.scatter(x, y, color=color_map[category], s=150, alpha=0.8, edgecolors='black')
    ax2.text(x, y, str(i), fontsize=11, fontweight='bold', ha='center', va='center')

ax2.set_xlabel('t-SNE 1')
ax2.set_ylabel('t-SNE 2')
ax2.set_title('2D t-SNE Visualization', fontsize=14)
ax2.grid(True, alpha=0.3)

# Add legend
legend_elements = [plt.Line2D([0], [0], marker='o', color='w',
                              markerfacecolor=color_map[cat],
                              markersize=10,
                              label=cat,
                              markeredgecolor='black')
                  for cat in unique_categories]
fig.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, 0.05),
           ncol=len(unique_categories), fontsize=10)

plt.tight_layout(rect=[0, 0.1, 1, 0.95])
plt.show()

# Print summary of what each sentence is about
print("\n" + "="*60)
print("SENTENCE SUMMARY")
print("="*60)
for i, (text, cat) in enumerate(zip(source_texts, categories)):
    print(f"{i:2d} [{cat:15s}]: {text[:60]}...")

## 🎯 Chunk Retrieval for RAG

### What is RAG?
** Retrieval-Augmented Generation (RAG)** combines:

- **Retrieval**: Find relevant information from knowledge base

- **Augmentation**: Add context to the query

- **Generation**: Produce informed, accurate responses

In [ ]:
# Using sentence-transformers library
from sentence_transformers import SentenceTransformer
import numpy as np

# 1. Load model
model = SentenceTransformer('all-MiniLM-L6-v2')  # 384 dimensions

# 2. Prepare chunks
chunks = [
    "PyTorch is a machine learning framework based on the Torch library.",
    "The average cat lifespan is between 13-17 years",
    "You're a wizard, Harry.",
    "The Eiffel Tower is in Paris, France.",
    "Paris is the capital city of France.",
    "France is known for its wine and cuisine.",
    "The Louvre Museum houses the Mona Lisa."
    "Space, the final frontier.",
    "I feel 😀",
    "Le meilleur moyen de bien commencer chaque journée est : à son réveil, de réfléchir si l'on ne peut pas ce jour-là faire plaisir au moins à une personne.",
    "I'm going to make him an offer he can't refuse.",
    "La joie est une émotion profonde et contagieuse qui nous inspire à vivre pleinement chaque jour.",
    "真正的快乐是内在的，它只有在人类的心灵里才能发现。",
    "Finding Coffee Spots: For your caffeine fix, head to the break room's coffee machine or cross the street to the café for artisan coffee.",
    "Team-Building Activities: We foster team spirit with monthly outings and weekly game nights. Feel free to suggest new activity ideas anytime!",
    "Working Hours Flexibility: We prioritize work-life balance. While our core hours are 9 AM to 5 PM, we offer flexibility to adjust as needed."
]

# 3. Embed all chunks (batch processing)
chunk_embeddings = model.encode(chunks, normalize_embeddings=True)
# shape: (4, 384) - 4 chunks, 384 dimensions each

# 4. Embed user query
query = "Happy"
query_embedding = model.encode([query], normalize_embeddings=True)[0]
print(f"Query Embedding (First 5 floats): '{query_embedding[:5]}'")

# 5. Find most relevant chunk
similarities = np.dot(chunk_embeddings, query_embedding)
# Might get: [0.45, 0.92, 0.31, 0.28]
# Chunk 2 wins with "Paris is the capital city of France."


# Find the index of the maximum dot product
most_similar_index = np.argmax(similarities)

# Get the most similar source text
most_similar_text = chunks[most_similar_index]

print(f"The most similar text to '{query}' is:")
print(most_similar_text)

top_k = 6
top_indices = np.argsort(similarities)[-top_k:][::-1]  # Descending order

print(f"Top {top_k} relevant chunks for query: '{query}'")
print("=" * 60)

# Display results with scores
for i, idx in enumerate(top_indices, 1):
    similarity_score = similarities[idx]
    print(f"\n{i}. Chunk {idx} (Similarity: {similarity_score:.4f}):")
    print(f"   {chunks[idx]}")

## 📈 Visualizing Retrieval Results

In [ ]:
import matplotlib.pyplot as plt
# Add threshold filtering
THRESHOLD = 0.15
relevant_indices = [idx for idx in top_indices if similarities[idx] > THRESHOLD]

# Format for LLM context
context_chunks = [chunks[idx] for idx in relevant_indices]
context = "\n\n".join([f"[Source {i+1}] {text}" for i, text in enumerate(context_chunks)])

print("\nFiltered Context for LLM:")
print("=" * 60)
print(context)

# Create visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart of all similarities
ax1.bar(range(len(similarities)), similarities, alpha=0.6)
ax1.set_title('Similarity Scores for All Chunks')
ax1.set_xlabel('Chunk Index')
ax1.set_ylabel('Cosine Similarity')
ax1.axhline(y=THRESHOLD, color='r', linestyle='--', label=f'Threshold ({THRESHOLD})')

# Highlight top 4
for idx in top_indices:
    ax1.bar(idx, similarities[idx], color='orange')

# Top 4 detailed view
y_pos = range(len(top_indices))
ax2.barh(y_pos, [similarities[idx] for idx in top_indices], color='green')
ax2.set_yticks(y_pos)
ax2.set_yticklabels([f'Chunk {idx}' for idx in top_indices])
ax2.set_xlabel('Cosine Similarity')
ax2.set_title('Top 4 Most Relevant Chunks')
ax2.axvline(x=THRESHOLD, color='r', linestyle='--')

plt.tight_layout()
plt.show()

## 🚀 End-to-End RAG with Re-ranking

### Why Two-Stage Retrieval?
1- **First Stage (Fast)**: Find many potentially relevant documents

2- **Second Stage (Accurate)**: Re-rank with more sophisticated model

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np
from typing import List, Tuple

# 1. Charger les modèles
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')  # Premier stage
reranker_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')  # Reranker

# 2. Préparer les chunks
chunks = [
    "PyTorch is a machine learning framework based on the Torch library.",
    "The average cat lifespan is between 13-17 years",
    "You're a wizard, Harry.",
    "The Eiffel Tower is in Paris, France.",
    "Paris is the capital city of France.",
    "France is known for its wine and cuisine.",
    "The Louvre Museum houses the Mona Lisa.",
    "Space, the final frontier.",
    "I feel 😀",
    "I'm going to make him an offer he can't refuse.",
    "Today I am very happy",
    "La joie est une émotion profonde et contagieuse qui nous inspire à vivre pleinement chaque jour.",
    "真正的快乐是内在的，它只有在人类的心灵里才能发现。",
    "Joining Slack Channels: You will receive an invite via email. Be sure to join relevant channels to stay informed and engaged.",
    "Finding Coffee Spots: For your caffeine fix, head to the break room's coffee machine or cross the street to the café for artisan coffee.",
    "Team-Building Activities: We foster team spirit with monthly outings and weekly game nights. Feel free to suggest new activity ideas anytime!",
    "Working Hours Flexibility: We prioritize work-life balance. While our core hours are 9 AM to 5 PM, we offer flexibility to adjust as needed."
]

# 3. Premier stage: Embedding + recherche vectorielle
def first_stage_retrieval(query: str, chunks: List[str], top_k: int = 10):
    """Première étape: recherche approximative avec embeddings"""
    # Encoder les chunks et la query
    chunk_embeddings = embedding_model.encode(chunks, normalize_embeddings=True)
    query_embedding = embedding_model.encode([query], normalize_embeddings=True)[0]
    
    # Calculer les similarités cosinus
    similarities = np.dot(chunk_embeddings, query_embedding)
    
    # Récupérer les top_k indices
    top_indices = np.argsort(similarities)[-top_k:][::-1]
    
    # Retourner les chunks et leurs scores
    first_stage_results = []
    for idx in top_indices:
        first_stage_results.append({
            'chunk': chunks[idx],
            'first_stage_score': float(similarities[idx]),
            'index': idx
        })
    
    return first_stage_results

# 4. Second stage: Reranking avec cross-encoder
def rerank_results(query: str, first_stage_results: List[dict], top_k: int = 4):
    """Deuxième étape: reranking précis avec cross-encoder"""
    # Préparer les paires (query, chunk) pour le cross-encoder
    pairs = [(query, result['chunk']) for result in first_stage_results]
    
    # Obtenir les scores de reranking
    rerank_scores = reranker_model.predict(pairs)
    
    # Combiner les scores
    for i, result in enumerate(first_stage_results):
        result['rerank_score'] = float(rerank_scores[i])
        # Score combiné (pondéré)
        result['combined_score'] = 0.3 * result['first_stage_score'] + 0.7 * result['rerank_score']
    
    # Trier par score combiné
    reranked_results = sorted(first_stage_results, key=lambda x: x['combined_score'], reverse=True)
    
    # Retourner les top_k
    return reranked_results[:top_k]

# 5. Pipeline complet
def full_rag_pipeline(query: str, chunks: List[str], 
                      first_stage_k: int = 10, 
                      final_k: int = 4):
    """Pipeline RAG complet avec deux étages"""
    
    # Étape 1: Recherche approximative
    print("🎯 Étape 1: Recherche vectorielle (first-stage retrieval)")
    first_stage_results = first_stage_retrieval(query, chunks, top_k=first_stage_k)
    
    print(f"\n📊 Top {first_stage_k} résultats du premier étage:")
    for i, result in enumerate(first_stage_results[:5], 1):
        print(f"{i}. [Score: {result['first_stage_score']:.4f}] {result['chunk'][:80]}...")
    
    # Étape 2: Reranking
    print(f"\n🎯 Étape 2: Reranking avec cross-encoder")
    final_results = rerank_results(query, first_stage_results, top_k=final_k)
    
    print(f"\n✅ Top {final_k} résultats après reranking:")
    for i, result in enumerate(final_results, 1):
        print(f"\n{i}. Chunk {result['index']}")
        print(f"   Score 1er étage: {result['first_stage_score']:.4f}")
        print(f"   Score rerank:    {result['rerank_score']:.4f}")
        print(f"   Score combiné:   {result['combined_score']:.4f}")
        print(f"   Texte: {result['chunk']}")
    
    return final_results

# 6. Exécuter
query = "Happy"
print(f"🔍 Query: '{query}'")
print("=" * 80)

results = full_rag_pipeline(query, chunks)

# 7. Préparer le contexte pour LLM
context_chunks = [r['chunk'] for r in results]
context = "\n\n".join([f"[Chunk {i+1}] {text}" for i, text in enumerate(context_chunks)])

print("\n📋 Contexte formaté pour LLM:")
print("=" * 80)
print(context)


## 🏆 Practical Applications & Best Practices

This is a code from scratch to understand embedding and RAG. More sophisticated frameworkds and Vector DB exists.

### Real-World Use Cases
- **Document Search**: Find relevant documents in knowledge bases

- **Customer Support**: Retrieve FAQ answers based on customer queries

- **Content Recommendation**: Suggest similar articles or products

- **Semantic Search**: Go beyond keyword matching to understand intent